### Configuration

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import copy

SEED = 0

In [2]:
# Network Parameters
XM = 100
YM = 100
SINK_X = 50
SINK_Y = 50
NUM_NODES = 100
INITIAL_ENERGY = 0.5
MIN_ENERGY = 0.05
ROUNDS = 60000

# Traffic Constraints
DATA_GENERATION_PROB = 0.09
COMM_RANGE = 87.0            # d0 crossover

# [cite_start]Energy Model [cite: 391]
E_ELEC = 50e-9
E_FS = 10e-12
E_MP = 0.0013e-12
E_DA = 5e-9

# Packet Sizes (Critical for Energy Realism)
PACKET_SIZE_DATA = 4000
PACKET_SIZE_CTRL = 200     # Overhead packets
E_IDLE_PER_ROUND = 50e-6   # Constant drain

# Optimization Constants
NUM_CLUSTERS = 5
FUZZY_M = 2.0
CHBCO_POP_SIZE = 15
CHBCO_MAX_ITER = 20
W1, W2, W3 = 0.35, 0.35, 0.30

In [3]:
class Node:
    def __init__(self, id, x, y):
        self.id = id
        self.x = x
        self.y = y
        self.energy = INITIAL_ENERGY
        self.alive = True
        self.cluster_id = -1
        self.is_ch = False
        self.dist_to_sink = np.sqrt((x - SINK_X)**2 + (y - SINK_Y)**2)

    def consume_energy(self, amount):
        if self.alive:
            self.energy -= amount
            if self.energy <= MIN_ENERGY:
                self.energy = 0
                self.alive = False

In [4]:
def calculate_distance(n1, n2):
    return np.sqrt((n1.x - n2.x)**2 + (n1.y - n2.y)**2)


def chebyshev_distance(coords, center):
    # [cite_start]Eq (5) [cite: 221]
    return np.maximum(np.abs(coords[:, 0] - center[0]), np.abs(coords[:, 1] - center[1]))


def check_packet_loss(dist):
    # Realistic Loss Model
    prob_loss = min(0.2, 0.1 + 0.3 * (dist / COMM_RANGE))
    return random.random() < prob_loss


def calculate_tx_energy(dist, bits):
    # [cite_start]Eq (1) [cite: 201]
    if dist <= COMM_RANGE:
        return (E_ELEC * bits) + (E_FS * bits * (dist**2))
    else:
        return (E_ELEC * bits) + (E_MP * bits * (dist**4))


def calculate_rx_energy(bits):
    return E_ELEC * bits

### FAULT TOLERANCE

In [5]:
def get_fault_tolerance_cost(candidate, nodes, ch_ids_prev):
    """
    [cite_start]Calculates Cost for Fault Recovery based on Eq (34)[cite: 384].
    Cost = EC_si_ai * EC_ai_sink * NCCR * NBPR * dist
    """
    # EC terms: Ratio of Available to Residual (Simplified to normalized Energy)
    ec_node = candidate.energy / INITIAL_ENERGY
    ec_sink = 1.0 / (candidate.dist_to_sink + 1e-5)

    # NBPR: Backward CH count (Bonus if it was a CH before)
    nbpr = 1.5 if candidate.id in ch_ids_prev else 1.0

    # Dist: Inverse distance preference (Higher cost = Better candidate)
    dist_score = 1.0 / (candidate.dist_to_sink + 1e-5)

    cost = ec_node * ec_sink * nbpr * dist_score
    return cost

In [ ]:
def fault_tolerance_check(nodes, ch_ids, clusters, ch_ids_prev):
    """
    [cite_start]Implements Algorithm 2: Fault Tolerance Mechanism[cite: 369].
    Detects dead CHs and reassigns using Cost Function.
    """
    final_ch_ids = []
    reassigned_map = {}  # Maps Old_CH -> New_CH

    for ch_id in ch_ids:
        ch_node = nodes[ch_id]
        # Detection: Check Energy/Alive status
        if ch_node.alive and ch_node.energy > MIN_ENERGY:
            final_ch_ids.append(ch_id)
            reassigned_map[ch_id] = ch_id
        else:
            # [cite_start]Fault Recovery [cite: 380]
            members = clusters.get(ch_node.cluster_id, [])
            best_cand = -1
            max_cost = -1.0

            for mid in members:
                if mid == ch_id:
                    continue
                mnode = nodes[mid]
                if not mnode.alive or mnode.energy <= MIN_ENERGY:
                    continue

                # Eq (34)
                cost = get_fault_tolerance_cost(mnode, nodes, ch_ids_prev)
                if cost > max_cost:
                    max_cost = cost
                    best_cand = mid

            if best_cand != -1:
                nodes[best_cand].is_ch = True
                final_ch_ids.append(best_cand)
                reassigned_map[ch_id] = best_cand

    return final_ch_ids, reassigned_map

### Clustering - Routing

In [7]:
def run_ifcm_clustering(nodes, alive_indices):
    np.random.seed(SEED)
    # Improved Fuzzy C-Means [cite: 207]
    n_samples = len(alive_indices)
    if n_samples < NUM_CLUSTERS:
        return {0: alive_indices}, np.array([[SINK_X, SINK_Y]])

    X = np.array([[nodes[i].x, nodes[i].y] for i in alive_indices])
    U = np.random.rand(n_samples, NUM_CLUSTERS)
    U = U / U.sum(axis=1, keepdims=True)
    centers = np.zeros((NUM_CLUSTERS, 2))

    for _ in range(10):
        um = U ** FUZZY_M
        num = um.T @ X
        den = um.T.sum(axis=1, keepdims=True)
        centers = num / (den + 1e-10)

        # Eq (5) Chebyshev Distance Usage
        dists = np.zeros((n_samples, NUM_CLUSTERS))
        for j in range(NUM_CLUSTERS):
            dists[:, j] = chebyshev_distance(X, centers[j])

        dists = np.fmax(dists, 1e-10)
        inv = 1.0 / dists
        pow_t = 2.0 / (FUZZY_M - 1)
        sum_inv = np.sum(inv ** pow_t, axis=1, keepdims=True)
        new_U = (inv ** pow_t) / (sum_inv + 1e-10)

        if np.allclose(U, new_U, atol=1e-2):
            break
        U = new_U

    clusters = {j: [] for j in range(NUM_CLUSTERS)}
    labels = np.argmax(U, axis=1)
    for idx, label in enumerate(labels):
        nodes[alive_indices[idx]].cluster_id = label
        clusters[label].append(alive_indices[idx])

    return clusters, centers

In [8]:
def select_cluster_heads(nodes, clusters, centers):
    # Section 3.2: Node with shortest distance to center is CH
    ch_ids = []
    for cid, members in clusters.items():
        if not members:
            continue
        center = centers[cid]
        best, min_d = -1, float('inf')

        for nid in members:
            nodes[nid].is_ch = False
            d = np.sqrt((nodes[nid].x - center[0])**2 +
                        (nodes[nid].y - center[1])**2)
            if d < min_d and nodes[nid].energy > MIN_ENERGY:
                min_d = d
                best = nid

        if best != -1:
            nodes[best].is_ch = True
            ch_ids.append(best)
    return ch_ids

In [ ]:
class CHBCO_Optimizer:
    # Custom Honey Badger and Coot Optimization [cite: 243]
    def __init__(self, ch_ids, nodes):
        self.ch_ids = ch_ids
        self.nodes = nodes
        self.dim = len(ch_ids)
        self.lb, self.ub = 0, self.dim + 0.99

    def decode(self, pos):
        routes = {}
        for i, val in enumerate(pos):
            idx = int(np.floor(val))
            idx = max(0, min(idx, self.dim))
            curr = self.ch_ids[i]
            if idx == self.dim:
                routes[curr] = -1
            else:
                nxt = self.ch_ids[idx]
                routes[curr] = -1 if nxt == curr else nxt
        return routes

    def fitness(self, pos):
        # Eq (33) Objective Function [cite: 364]
        routes = self.decode(pos)
        e_sum, d_sum, lq_sum = 0, 0, 0
        pen = 0
        for src, dst in routes.items():
            n_src = self.nodes[src]
            e_sum += (1.0 - n_src.energy/INITIAL_ENERGY)
            if dst == -1:
                d = n_src.dist_to_sink
            else:
                d = calculate_distance(n_src, self.nodes[dst])
                if routes.get(dst) == src:
                    pen += 50
            d_sum += d
            lq_sum += (1.0 - (1.0/(1.0 + 0.1*d)))

        return W1*(e_sum/self.dim) + W2*(d_sum/(self.dim*141)) + W3*(lq_sum/self.dim) + pen

    def optimize(self):
        np.random.seed(SEED)
        random.seed(SEED)
        X = np.random.uniform(self.lb, self.ub, (CHBCO_POP_SIZE, self.dim))
        fit = np.array([self.fitness(x) for x in X])
        best_idx = np.argmin(fit)
        X_prey, f_prey = X[best_idx].copy(), fit[best_idx]

        C = 2
        for t in range(1, CHBCO_MAX_ITER+1):
            alpha = C * np.exp(-t/CHBCO_MAX_ITER)
            for i in range(CHBCO_POP_SIZE):
                di = np.linalg.norm(X[i] - X_prey) + 1e-10
                I = random.random() * \
                    (np.linalg.norm(X[i] - X[(i+1) %
                     CHBCO_POP_SIZE])**2) / (4*np.pi*di**2)

                # Hybrid HBA/Coot Logic (Eq 11-20)
                if random.random() <= 0.5:
                    F = 1 if random.random() <= 0.5 else -1
                    t1 = X_prey + F*alpha*I*X_prey
                    t2 = F*random.random()*alpha*di*abs(math.cos(2*np.pi*random.random())
                                                        * (1-math.cos(2*np.pi*random.random())))
                    X_new = t1 + t2
                else:
                    # Improved Honey Phase (Eq 32)
                    R1, R = random.random(), random.random()
                    denom = 1 + 2*R1*math.cos(2*np.pi*R)
                    if abs(denom) < 1e-5:
                        denom = 1e-5
                    X_new = (
                        X_prey * denom - random.choice([-1, 1]) * random.random() * alpha * di) / denom

                X_new = np.clip(X_new, self.lb, self.ub)
                f_new = self.fitness(X_new)
                if f_new <= fit[i]:
                    X[i], fit[i] = X_new, f_new
                    if f_new <= f_prey:
                        X_prey, f_prey = X_new.copy(), f_new

        return self.decode(X_prey)

### Simulation

In [10]:
def run_simulation():
    random.seed(SEED)
    nodes = [Node(i, random.uniform(0, XM), random.uniform(0, YM))
             for i in range(NUM_NODES)]
    ch_ids_prev = []  # Required for Eq 34 (NBPR)

    total_gen, total_del = 0, 0
    fnd, hnd, lnd = None, None, None
    stats_alive, stats_energy = [], []

    for r in range(1, ROUNDS + 1):
        alive_nodes = [n for n in nodes if n.alive]
        alive_indices = [n.id for n in alive_nodes]
        n_alive = len(alive_nodes)

        # Metrics
        dead = NUM_NODES - n_alive
        if dead > 0 and fnd is None:
            fnd = r
        if dead >= NUM_NODES/2 and hnd is None:
            hnd = r
        if n_alive == 0:
            if lnd is None:
                lnd = r
            break

        stats_alive.append(n_alive)
        stats_energy.append(np.mean([n.energy for n in nodes]))

        # PHASE 1: SETUP & CLUSTERING
        # 1. Hello Packets (Control Overhead)
        for n in alive_nodes:
            e_tx = calculate_tx_energy(COMM_RANGE/2, PACKET_SIZE_CTRL)
            e_rx = calculate_rx_energy(PACKET_SIZE_CTRL) * 5
            n.consume_energy(e_tx + e_rx)

        # 2. IFCM Clustering
        clusters, centers = run_ifcm_clustering(nodes, alive_indices)
        raw_ch_ids = select_cluster_heads(nodes, clusters, centers)

        # 3. CH Advertisement
        for ch in raw_ch_ids:
            if nodes[ch].alive:
                nodes[ch].consume_energy(
                    calculate_tx_energy(COMM_RANGE, PACKET_SIZE_CTRL))

        # 4. Join Requests
        for nid in alive_indices:
            if nid not in raw_ch_ids:
                my_ch = -1
                lbl = nodes[nid].cluster_id
                for c in raw_ch_ids:
                    if nodes[c].cluster_id == lbl:
                        my_ch = c

                if my_ch != -1 and nodes[my_ch].alive:
                    d = calculate_distance(nodes[nid], nodes[my_ch])
                    nodes[nid].consume_energy(
                        calculate_tx_energy(d, PACKET_SIZE_CTRL))
                    nodes[my_ch].consume_energy(
                        calculate_rx_energy(PACKET_SIZE_CTRL))

        # PHASE 2: FAULT TOLERANCE & ROUTING

        # 5. Fault Tolerance (Algorithm 2)
        # Replaces dead CHs with best candidates using Eq 34
        active_ch, ch_map = fault_tolerance_check(
            nodes, raw_ch_ids, clusters, ch_ids_prev)
        ch_ids_prev = active_ch[:]  # Update history

        # Update clusters based on re-election
        final_clusters = {}
        for old_ch, new_ch in ch_map.items():
            lbl = nodes[old_ch].cluster_id
            final_clusters[new_ch] = clusters.get(lbl, [])

        # FIX: Crash Prevention
        if not active_ch:
            for n in alive_nodes:
                n.consume_energy(E_IDLE_PER_ROUND)
            continue

        # 6. Routing Optimization
        optimizer = CHBCO_Optimizer(active_ch, nodes)
        routes = optimizer.optimize()

        # PHASE 3: DATA TRANSMISSION
        packets_at_ch = {ch: 0 for ch in active_ch}

        # A. Members -> CH
        for ch in active_ch:
            members = final_clusters.get(ch, [])
            for mid in members:
                if mid == ch or not nodes[mid].alive:
                    continue

                if random.random() > DATA_GENERATION_PROB:
                    continue

                total_gen += 1
                dist = calculate_distance(nodes[mid], nodes[ch])
                nodes[mid].consume_energy(
                    calculate_tx_energy(dist, PACKET_SIZE_DATA))

                if not check_packet_loss(dist) and nodes[ch].alive:
                    nodes[ch].consume_energy(
                        calculate_rx_energy(PACKET_SIZE_DATA))
                    packets_at_ch[ch] += 1

        # B. CH -> Sink
        for ch in active_ch:
            if not nodes[ch].alive:
                continue

            # CH own data
            if random.random() <= DATA_GENERATION_PROB:
                total_gen += 1
                packets_at_ch[ch] += 1

            if packets_at_ch[ch] == 0:
                continue

            # Aggregation Energy
            nodes[ch].consume_energy(
                E_DA * PACKET_SIZE_DATA * packets_at_ch[ch])

            # Multi-hop Transmission
            curr = ch
            delivered = False
            hops = 0
            dropped = False

            while hops < 8:
                nxt = routes.get(curr, -1)
                dist = nodes[curr].dist_to_sink if nxt == - \
                    1 else calculate_distance(nodes[curr], nodes[nxt])

                nodes[curr].consume_energy(
                    calculate_tx_energy(dist, PACKET_SIZE_DATA))

                if check_packet_loss(dist):
                    dropped = True
                    break

                if nxt == -1:
                    delivered = True
                    break
                else:
                    if nodes[nxt].alive:
                        nodes[nxt].consume_energy(
                            calculate_rx_energy(PACKET_SIZE_DATA))
                        curr = nxt
                        hops += 1
                    else:
                        dropped = True
                        break

            if delivered and not dropped:
                total_del += packets_at_ch[ch]

        for n in alive_nodes:
            n.consume_energy(E_IDLE_PER_ROUND)

        # if r % 100 == 0:
        #     print(
        #         f"Round {r}: Alive={n_alive} | Energy={np.mean([n.energy for n in nodes]):.4f} | Gen={total_gen} | Del={total_del}")

    # Results
    pdr = (total_del / total_gen * 100) if total_gen > 0 else 0
    print(
        f"\nFINAL RESULTS:\nPDR: {pdr:.2f}%\nTotal Packets Generadet: {total_gen}\nTotal Packets Delivered: {total_del}\nFND: {fnd}\nHND: {hnd}\nLND: {lnd}")

    # plt.figure(figsize=(10, 4))
    # plt.subplot(1, 2, 1)
    # plt.plot(stats_alive)
    # plt.title("Alive Nodes")
    # plt.subplot(1, 2, 2)
    # plt.plot(stats_energy, color='red')
    # plt.title("Avg Energy")
    # plt.show()

### RUN

In [11]:
print(f"EACBR Simulation")
for i in range(31):
    SEED = i
    print(f"\nSEED {SEED}")
    run_simulation()

EACBR Simulation

SEED 0

FINAL RESULTS:
PDR: 68.18%
Total Packets Generadet: 21526
Total Packets Delivered: 14677
FND: 334
HND: 2684
LND: 3540

SEED 1

FINAL RESULTS:
PDR: 70.75%
Total Packets Generadet: 15888
Total Packets Delivered: 11240
FND: 412
HND: 2782
LND: 3560

SEED 2

FINAL RESULTS:
PDR: 72.09%
Total Packets Generadet: 24513
Total Packets Delivered: 17672
FND: 291
HND: 2532
LND: 3409

SEED 3

FINAL RESULTS:
PDR: 89.31%
Total Packets Generadet: 6471
Total Packets Delivered: 5779
FND: 527
HND: 3297
LND: 3610

SEED 4

FINAL RESULTS:
PDR: 70.09%
Total Packets Generadet: 25727
Total Packets Delivered: 18031
FND: 352
HND: 2588
LND: 3225

SEED 5

FINAL RESULTS:
PDR: 65.34%
Total Packets Generadet: 17234
Total Packets Delivered: 11261
FND: 413
HND: 2845
LND: 3561

SEED 6

FINAL RESULTS:
PDR: 67.59%
Total Packets Generadet: 23215
Total Packets Delivered: 15692
FND: 357
HND: 2748
LND: 3379

SEED 7

FINAL RESULTS:
PDR: 66.43%
Total Packets Generadet: 14106
Total Packets Delivered: 9371